# SwinV2-Tiny / E1-B1 — Cartesian iris annulus; eyelashes retained

All implementation code is visible below. This notebook does not import another project file.
The three training seeds are independent pretrained initializations. No test image, pair or score enters training or checkpoint selection.

Input **[B,1,320,320]**; one fully fine-tuned shared encoder, 256-dimensional unit-normalized output.
Subject partitions use dataset+subject; physical-eye labels use dataset+subject+eye. Both eyes stay together.
CASIA-Lamp is outside this protocol. Native masks were generated with the published Worldcoin iris semantic model.
E1-C uses the frozen V2 estimator only; unavailable cases stay unavailable.

Defaults are intentionally opt-in: run all cells to inspect code and recorded results; set `RUN_TRAINING=True` to reproduce this condition's three runs.
See [asset instructions](../../ASSETS.md) before enabling data access.

## Imports

Run this cell first. All package imports are collected here. ONNX Runtime is optional unless you explicitly generate native masks; saved-mask workflows still run without it.

In [ ]:
from __future__ import annotations

# Standard library
import copy
import gc
import hashlib
import json
import math
import os
import platform
import random
import time
import uuid
from collections import Counter, defaultdict
from collections.abc import Sequence
from contextlib import contextmanager
from copy import deepcopy
from dataclasses import dataclass
from decimal import Decimal
from importlib import metadata
from numbers import Integral
from pathlib import Path, PureWindowsPath
from typing import Any, Mapping

# Scientific libraries, models and notebook display
import torch
import timm
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Same backend as the saved scientific figures.
import matplotlib.pyplot as plt
from IPython.display import Image, display
from sklearn.metrics import auc, roc_curve
from timm.models.swin_transformer_v2 import WindowAttention
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Sampler

# Optional: only explicit native-mask generation needs ONNX Runtime.
_ONNXRUNTIME_IMPORT_ERROR = None
try:
    import onnxruntime as ort
except (ImportError, OSError) as error:
    ort = None
    _ONNXRUNTIME_IMPORT_ERROR = error

## 1. Editable paths and execution switches

In [ ]:
# Set REPOSITORY_ROOT explicitly when running a copied notebook outside this checkout.
REPOSITORY_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                       if (p / "Assets/subject_splits.json").is_file()), Path.cwd())
DATA_ROOT = (REPOSITORY_ROOT / "data").resolve()  # edit: licensed asset bundle
OUTPUT_ROOT = Path("reproduced_outputs").resolve()     # never recorded history directories
IMAGE_MANIFEST = DATA_ROOT / "image_manifest.parquet"
SUBJECT_SPLITS = REPOSITORY_ROOT / "Assets/subject_splits.json"
IMAGE_EXCLUSIONS = REPOSITORY_ROOT / "Assets/image_exclusions.json"
VALIDATION_PAIR_FILES = {d: REPOSITORY_ROOT / "Assets/Validation Pairs" / f"validation_pairs_{n}.parquet"
    for d,n in [("CUST-Iris","cust_iris"),("CASIA-Interval","casia_interval"),("IITD","iitd")]}
PRETRAINED_ASSET = DATA_ROOT / "pretrained/swinv2_tiny_window8_256.ms_in1k.safetensors"
PATH_REMAP = {}   # explicit prefix relocation only, if your metadata needs it
ARCHITECTURE = 'SwinV2-Tiny'
CONDITION = 'E1-B1'
SEEDS = [42, 123, 2025]
DEVICE = "cuda"
ALLOW_PRETRAINED_DOWNLOAD = False
RUN_TRAINING = False
CHECK_REAL_ASSETS = False
RUN_MODEL_SMOKE = False
RESUME_NOTEBOOK_RUNS = False
SHOW_RECORDED_RESULTS = True

## 2. Complete implementation

Only standard third-party libraries are imported. The frozen mathematical functions below are transcribed from the validated implementation.
Section prefixes prevent name collisions while keeping every definition in this notebook. There is no source loader, private integrity gate or code restoration step.

The global raw-training normalization is fixed at mean **0.48287347719744544**, SD **0.2741831329802273**.
Cartesian images use whole-frame aspect-preserving 320×320 letterboxing; caches are never resized again.
Binary supports use nearest-exact interpolation. Augmentation precedes Cartesian masking.
Polar images use 64×512 native sampling with independent pupil/iris centers; C2 reuses C1 coordinates and removes the active bilinear eyelash footprint.
Frozen caches may be supplied directly; preprocessing is never silently rerun.

### Scientific settings and environment

In [ ]:
config_DOMAINS = ('CUST-Iris', 'CASIA-Interval', 'IITD')
config_EXTERNAL_DOMAINS = ('AMF', 'UTIRIS_NIR')
config_CONDITIONS = ('E1-A', 'E1-B1', 'E1-B2', 'E1-C1', 'E1-C2')
config_ARCHITECTURES = ('ResNet-18', 'SwinV2-Tiny')
config_SEEDS = (42, 123, 2025)
config_NORMALIZATION = {'mean': 0.48287347719744544, 'std': 0.2741831329802273}

def config_frozen_config(architecture, condition, seed=42, device='cuda'):
    if architecture not in config_ARCHITECTURES or condition not in config_CONDITIONS or seed not in config_SEEDS:
        raise ValueError('Choose the original backbone, E1 condition, and seed 42/123/2025')
    path = None
    config = deepcopy(config_FROZEN_DEFAULTS[architecture][condition])
    config.update(architecture_label=architecture, condition=condition, seed=int(seed))
    config['runtime']['device'] = str(device)
    config['input_shape'] = [1, 64, 512] if condition.startswith('E1-C') else [1, 320, 320]
    config['normalization'] = dict(config_NORMALIZATION)
    return config

def config_environment_info():
    """Display installed versions; do not install packages or change the environment."""
    packages = {}
    for name in ('torch', 'torchvision', 'timm', 'numpy', 'opencv-python', 'pandas', 'pyarrow', 'scipy', 'scikit-learn', 'nbformat'):
        try:
            packages[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            raise ImportError(f'Missing {name}. Install requirements.txt into the selected notebook kernel environment.') from None
    return {'python': platform.python_version(), **packages, 'cuda_build': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
config_FROZEN_DEFAULTS = {'ResNet-18': {'E1-A': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'raw_ocular'}}, 'E1-B1': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'annulus'}}, 'E1-B2': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'visible_iris'}}, 'E1-C1': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c1', 'eligibility_policy': 'pure_v2_common_available'}}, 'E1-C2': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c2', 'eligibility_policy': 'pure_v2_common_available'}}}, 'SwinV2-Tiny': {'E1-A': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [320, 320]}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'raw_ocular'}}, 'E1-B1': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [320, 320]}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'annulus'}}, 'E1-B2': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [320, 320]}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'visible_iris'}}, 'E1-C1': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [64, 512]}, 'data': {'eligibility_policy': 'pure_v2_common_available', 'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c1'}}, 'E1-C2': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [64, 512]}, 'data': {'eligibility_policy': 'pure_v2_common_available', 'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c2'}}}}

### Photometric augmentation and geometry errors

In [ ]:
class geometry_support_GeometryFailure(ValueError):

    def __init__(self, reason, detail=None):
        self.reason, self.detail = (reason, detail or {})
        super().__init__(reason)

### Photometric augmentation

In [ ]:
class augmentation_PhotometricAugmentation:
    """Probability0.5: multiplicative brightness and contrast jitter of±10%.

    Brightness is applied first and clamped to[0,1]. Contrast then scales about
    that image's mean, followed by another clamp. Three CPU Torch RNG draws per
    applied transform (gate, brightness, contrast) make checkpoint RNG sufficient.
    No interpolation, geometry, occlusion, erasing or channel changes occur.
    """

    def __init__(self, probability: float=0.5, brightness: float=0.1, contrast: float=0.1):
        if not 0 <= probability <= 1 or not 0 <= brightness <= 1 or (not 0 <= contrast <= 1):
            raise ValueError('Photometric settings must lie in[0,1]')
        self.probability, self.brightness, self.contrast = (probability, brightness, contrast)

    def __call__(self, image: torch.Tensor) -> torch.Tensor:
        if self.probability == 0 or torch.rand(()).item() >= self.probability:
            return image
        brightness = 1 + (2 * torch.rand(()).item() - 1) * self.brightness
        contrast = 1 + (2 * torch.rand(()).item() - 1) * self.contrast
        adjusted = (image * brightness).clamp(0, 1)
        mean = adjusted.mean()
        return ((adjusted - mean) * contrast + mean).clamp(0, 1)

def augmentation_augment_polar(image, validity, probability=0.5, brightness=0.1, contrast=0.1):
    """Photometric jitter uses valid tissue, then explicitly restores invalid zeroes."""
    if image.dtype != torch.float32 or validity.dtype != torch.bool or image.shape != validity.shape:
        raise ValueError('Expected matching float32 texture and Boolean polar validity')
    if not validity.any():
        raise geometry_support_GeometryFailure('no_valid_polar_augmentation_tissue')
    if torch.rand(()).item() < probability:
        bright = 1 + (2 * torch.rand(()).item() - 1) * brightness
        factor = 1 + (2 * torch.rand(()).item() - 1) * contrast
        image = (image * bright).clamp(0, 1)
        mean = image[validity].mean()
        image = ((image - mean) * factor + mean).clamp(0, 1)
    return image.masked_fill(~validity, 0.0)

### Native semantic masks: explicit Worldcoin inference, never silently regenerated

In [ ]:
segmentation_CLASSES = ('eyeball', 'iris', 'pupil', 'eyelashes', 'annulus', 'visible_iris')
segmentation_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
segmentation_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def segmentation_preprocess(gray: np.ndarray, height: int, width: int) -> np.ndarray:
    resized = cv2.resize(gray, (width, height), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(resized, cv2.COLOR_GRAY2RGB).astype(np.float32) / np.float32(255.0)
    return np.ascontiguousarray(((rgb - segmentation_MEAN) / segmentation_STD).transpose(2, 0, 1))

def segmentation_probability_masks(prediction: np.ndarray, height: int, width: int, threshold: float) -> dict[str, np.ndarray]:
    if prediction.ndim != 3 or prediction.shape[0] != 4:
        raise ValueError(f'Unexpected output shape: {prediction.shape}')
    if not np.isfinite(prediction).all() or prediction.min() < -1e-05 or prediction.max() > 1.00001:
        raise ValueError('Expected finite probabilities in [0, 1]; refusing to threshold logits')
    probabilities = cv2.resize(prediction.transpose(1, 2, 0), (width, height), interpolation=cv2.INTER_LINEAR)
    masks = {name: probabilities[:, :, index] > threshold for index, name in enumerate(segmentation_CLASSES[:4])}
    masks['annulus'] = masks['iris'] & ~masks['pupil']
    masks['visible_iris'] = masks['annulus'] & ~masks['eyelashes']
    return masks

def segmentation_make_session(model: Path, device: str, threads: int):
    if ort is None:
        raise ImportError(
            "Native-mask generation needs the optional ONNX Runtime dependency documented in ASSETS.md."
        ) from _ONNXRUNTIME_IMPORT_ERROR
    available = ort.get_available_providers()
    if device == 'cuda' and 'CUDAExecutionProvider' not in available:
        raise RuntimeError(f'CUDA requested but unavailable: {available}')
    providers = ['CPUExecutionProvider']
    if device != 'cpu' and 'CUDAExecutionProvider' in available:
        providers.insert(0, 'CUDAExecutionProvider')
    options = ort.SessionOptions()
    options.intra_op_num_threads = threads
    options.inter_op_num_threads = 1
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(str(model), sess_options=options, providers=providers)
    if device == 'cuda' and 'CUDAExecutionProvider' not in session.get_providers():
        raise RuntimeError('CUDA initialization failed; refusing an unexpected CPU fallback')
    inputs = session.get_inputs()
    if len(inputs) != 1 or inputs[0].type != 'tensor(float)':
        raise ValueError(f'Unsupported model inputs: {inputs}')
    shape = inputs[0].shape
    if len(shape) != 4 or shape[1] != 3 or (not all((isinstance(x, int) and x > 0 for x in shape[2:]))):
        raise ValueError(f'Expected NCHW input with fixed spatial dimensions: {shape}')
    print(f'ONNX Runtime {ort.__version__}; providers={session.get_providers()}; input={shape}', flush=True)
    return (session, inputs[0], ort.__version__)

def segmentation_segment_native(gray, session, model_input):
    if gray.ndim != 2 or gray.dtype != np.uint8:
        raise ValueError('Expected native uint8 grayscale')
    x = segmentation_preprocess(gray, *model_input.shape[2:])[None]
    prediction = session.run(None, {model_input.name: x})[0]
    if prediction.shape[0] != 1:
        raise ValueError('Expected one image prediction')
    return segmentation_probability_masks(prediction[0], *gray.shape, threshold=0.5)

### Image, mask and tensor transformations

In [ ]:
def preprocessing_read_gray(path, *, native=False):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing image: {path}. Configure DATA_ROOT/PATH_REMAP or supply the original private cached representation.')
    flags = cv2.IMREAD_GRAYSCALE | cv2.IMREAD_IGNORE_ORIENTATION if native else cv2.IMREAD_UNCHANGED
    image = cv2.imdecode(np.fromfile(path, np.uint8), flags)
    if image is None or image.dtype != np.uint8 or image.ndim != 2:
        raise ValueError(f'Expected unchanged uint8 grayscale pixels: {path}')
    return image

def preprocessing_read_binary(path, shape=None):
    mask = preprocessing_read_gray(path)
    if shape is not None and mask.shape != tuple(shape):
        raise ValueError(f'Image/mask geometry mismatch: {path}')
    if not np.isin(mask, [0, 255]).all():
        raise ValueError(f'Expected an original 0/255 binary mask; no threshold repair: {path}')
    return mask == 255

def preprocessing_geometry(width, height):
    scale = min(320 / width, 320 / height)
    rw, rh = (round(width * scale), round(height * scale))
    left, top = ((320 - rw) // 2, (320 - rh) // 2)
    return {'native_width': width, 'native_height': height, 'content_width': rw, 'content_height': rh, 'pad_left': left, 'pad_top': top, 'pad_right': 320 - rw - left, 'pad_bottom': 320 - rh - top, 'output_width': 320, 'output_height': 320}

def preprocessing_resize_raw(source, plan):
    dimensions = (plan['content_width'], plan['content_height'])
    content = source.copy() if source.shape == (dimensions[1], dimensions[0]) else cv2.resize(source, dimensions, interpolation=cv2.INTER_AREA)
    return cv2.copyMakeBorder(content, plan['pad_top'], plan['pad_bottom'], plan['pad_left'], plan['pad_right'], cv2.BORDER_CONSTANT, value=0)

def preprocessing_transform_support(mask, plan):
    if mask.dtype != np.bool_ or mask.shape != (plan['native_height'], plan['native_width']):
        raise ValueError('Native mask and raw-image geometry must match')
    dimensions = (plan['content_width'], plan['content_height'])
    content = mask.astype(np.uint8)
    if content.shape != (dimensions[1], dimensions[0]):
        content = cv2.resize(content, dimensions, interpolation=cv2.INTER_NEAREST_EXACT)
    return cv2.copyMakeBorder(content, plan['pad_top'], plan['pad_bottom'], plan['pad_left'], plan['pad_right'], cv2.BORDER_CONSTANT, value=0).astype(bool)

def preprocessing_native_cartesian(gray, masks=None):
    """Exact original letterbox and masks. Never stretch or crop around the iris."""
    plan = preprocessing_geometry(gray.shape[1], gray.shape[0])
    raw = preprocessing_resize_raw(gray, plan)
    if masks is None:
        return (raw, None, None)
    annulus = masks['iris'] & ~masks['pupil']
    return (raw, preprocessing_transform_support(annulus, plan), preprocessing_transform_support(annulus & ~masks['eyelashes'], plan))

def preprocessing_tensor_from_pixels(pixels, condition, normalization, *, support=None, validity=None, augmentation=None):
    image = torch.from_numpy(np.ascontiguousarray(pixels[None])).float().div_(255.0)
    if condition.startswith('E1-C'):
        if validity is None or validity.shape != pixels.shape or (not validity.any()) or pixels[~validity].any():
            raise ValueError('Polar pixels require matching nonempty validity and zero invalid pixels')
        if augmentation is not None:
            image = augmentation_augment_polar(image, torch.from_numpy(validity[None]), **augmentation)
    else:
        if augmentation is not None:
            image = augmentation_PhotometricAugmentation(**augmentation)(image)
        if condition in ('E1-B1', 'E1-B2'):
            if support is None or support.shape != pixels.shape or support.dtype != np.bool_:
                raise ValueError('Cartesian segmented inputs require the corresponding binary support')
            image = image.masked_fill(~torch.from_numpy(support)[None], 0.0)
    image = (image - normalization['mean']) / normalization['std']
    shape = (1, 64, 512) if condition.startswith('E1-C') else (1, 320, 320)
    if image.shape != shape or image.dtype != torch.float32 or (not torch.isfinite(image).all()):
        raise ValueError(f'Expected finite float32 input {shape}, got {tuple(image.shape)}')
    return image

### Frozen inventory, physical-eye identities and representation availability

In [ ]:
data_PATH_COLUMNS = ('native_path', 'raw_path', 'b1_support_path', 'b2_support_path', 'c1_path', 'c1_validity_path', 'c2_path', 'c2_validity_path', 'iris_mask_path', 'pupil_mask_path', 'eyelashes_mask_path')

def data_load_table(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing frozen metadata/pair table: {path}. Supply the documented CSV/Parquet asset; this workflow never invents an inventory or resamples a split.')
    if path.suffix.lower() == '.parquet':
        frame = pd.read_parquet(path)
    elif path.suffix.lower() == '.csv':
        frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    else:
        raise ValueError(f'Expected CSV or Parquet, got {path}')
    return frame

def data_explicit_bool(value, name):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value in ('true', 'True', '1', 1):
        return True
    if value in ('false', 'False', '0', 0):
        return False
    raise ValueError(f'{name} must be an explicit boolean, got {value!r}')

def data_present(value):
    return value is not None and (not (isinstance(value, float) and np.isnan(value))) and (str(value) != '')

def data_resolve_path(value, data_root, path_remap=None):
    if not data_present(value):
        return None
    text = str(value).replace('\\', '/')
    for old, new in sorted((path_remap or {}).items(), key=lambda item: -len(str(item[0]))):
        prefix = str(old).replace('\\', '/').rstrip('/')
        if text == prefix or text.startswith(prefix + '/'):
            return (Path(new).expanduser() / text[len(prefix):].lstrip('/')).resolve()
    path = Path(text).expanduser()
    if PureWindowsPath(text).is_absolute() and (not path.is_absolute()):
        raise ValueError(f'Foreign absolute path {text!r}. Set PATH_REMAP explicitly; no old user directory is assumed.')
    return (path if path.is_absolute() else Path(data_root) / path).resolve()

def data_load_inventory(manifest, *, data_root, subject_splits, image_exclusions, path_remap=None, strict_original_counts=True):
    """Validate frozen identity/split semantics, never file hashes or source seals."""
    frame = data_load_table(manifest)
    required = {'image_id', 'dataset', 'subject_id', 'eye_side', 'iris_id', 'source_id', 'split'}
    if (missing := (required - set(frame))):
        raise ValueError(f'Image manifest lacks {sorted(missing)}. See ASSETS.md for the portable schema; source_id must be the original canonical-content identifier.')
    frozen = json.loads(Path(subject_splits).read_text(encoding='utf-8'))
    exclusions = json.loads(Path(image_exclusions).read_text(encoding='utf-8'))
    if frozen['seed'] != 42 or set(frozen['datasets']) != set(config_DOMAINS):
        raise ValueError('Supply the original seed-42 subject allocations for the three primary datasets')
    if strict_original_counts:
        canonical = data_FROZEN_MEMBERSHIP

        def allocations(value):
            return {d: {s: sorted(map(str, ids)) for s, ids in splits.items()} for d, splits in value.items()}
        if allocations(frozen['datasets']) != allocations(canonical['datasets']):
            raise ValueError('Supplied labels change the original seed-42 subject allocations; no replacement split is permitted')
        if {r['image_id'] for r in exclusions['excluded_images']} != set(canonical['excluded_image_ids']) or {(r['dataset'], r['subject_id']) for r in exclusions['excluded_subjects']} != {(r['dataset'], r['subject_id']) for r in canonical['excluded_subjects']}:
            raise ValueError('Supplied image/subject exclusions differ from the original approved exclusions')
    assignment = {}
    for domain, partitions in frozen['datasets'].items():
        for split, subjects in partitions.items():
            for subject in subjects:
                key = (domain, str(subject))
                if key in assignment:
                    raise ValueError(f'Frozen subject occurs in multiple partitions: {key}')
                assignment[key] = split
    excluded_ids = {r['image_id'] for r in exclusions['excluded_images']}
    excluded_subjects = {(r['dataset'], r['subject_id']) for r in exclusions['excluded_subjects']}
    rows, excluded = ([], [])
    seen = set()
    for row in frame.to_dict('records'):
        for key in required:
            if not data_present(row[key]):
                raise ValueError(f'Missing {key}; labels/source identity may not be guessed')
            row[key] = str(row[key])
        if row['image_id'] in seen:
            raise ValueError(f"Duplicate image ID: {row['image_id']}")
        seen.add(row['image_id'])
        if row['eye_side'] not in ('L', 'R') or row['iris_id'] != f"{row['dataset']}:{row['subject_id']}:{row['eye_side']}":
            raise ValueError(f"Identity/eye namespace mismatch: {row['image_id']}")
        key = (row['dataset'], row['subject_id'])
        if row['image_id'] in excluded_ids or key in excluded_subjects:
            excluded.append(row['image_id'])
            continue
        if row['dataset'] in config_DOMAINS:
            if key not in assignment or row['split'] != assignment[key]:
                raise ValueError(f'Manifest changes the frozen dataset+subject partition: {key}')
        elif row['dataset'] in config_EXTERNAL_DOMAINS:
            if row['split'] != 'external':
                raise ValueError('AMF and UTIRIS_NIR must remain external, never training/validation')
        else:
            raise ValueError(f"Dataset outside E1 scope: {row['dataset']}")
        if 'included' in row and data_present(row['included']) and (not data_explicit_bool(row['included'], 'included')):
            raise ValueError(f"Unapproved additional exclusion: {row['image_id']}. Supply the retained frozen inventory, including QC flags.")
        for column in data_PATH_COLUMNS:
            row[column] = data_resolve_path(row.get(column), data_root, path_remap)
        row['quality_flags'] = row.get('quality_flags', [])
        rows.append(row)
    if not rows:
        raise ValueError('No retained images in the supplied frozen inventory')
    sources = defaultdict(list)
    for row in rows:
        sources[row['source_id']].append(row)
    duplicates = {source: group for source, group in sources.items() if len(group) > 1}
    if duplicates:
        raise ValueError('Redundant canonical source images remain in the included inventory; use the existing audited exact-duplicate exclusions, not a new deduplication rule')
    if strict_original_counts:
        expected = data_INVENTORY_COUNTS
        actual = Counter(((r['dataset'], r['split']) for r in rows if r['dataset'] in config_DOMAINS))
        required_counts = {(d, s): n for d, splits in expected['raw'].items() for s, n in splits.items()}
        if actual != required_counts:
            raise ValueError('Portable manifest is not the complete frozen primary inventory. Supply the original author metadata (all retained train/validation/test rows); do not invent a subset or replacement split. Synthetic software tests must explicitly disable this count check.')
        if 'c1_status' in frame and 'c2_status' in frame:
            actual = Counter(((r['dataset'], r['split']) for r in rows if r['dataset'] in config_DOMAINS and r['c1_status'] == r['c2_status'] == 'ok'))
            required_counts = {(d, s): n for d, splits in expected['polar_common'].items() for s, n in splits.items()}
            if actual != required_counts:
                raise ValueError('Pure-V2 common availability differs from the original audit; supply the frozen status/eligibility manifest rather than new exclusions')
    return (sorted(rows, key=lambda r: r['image_id']), {'images': len(rows), 'approved_exclusion_rows_filtered': excluded, 'groups': dict(Counter(((r['dataset'], r['split']) for r in rows))), 'new_subject_assignments': False, 'file_hash_checks': False})

def data_condition_view(records, condition, split):
    """Keep the historical common C1/C2 availability; report every failure."""
    if split not in ('train', 'validation', 'test', 'external'):
        raise ValueError('Unknown split')
    selected, unavailable = ([], [])
    for row in records:
        if row['split'] != split:
            continue
        if condition in ('E1-B1', 'E1-B2'):
            key = 'b1_status' if condition == 'E1-B1' else 'b2_status'
            if key in row and row[key] != 'ok':
                unavailable.append({'image_id': row['image_id'], 'reason': row[key]})
                continue
        if condition.startswith('E1-C'):
            if not all((name in row for name in ('c1_status', 'c2_status', 'geometry_version', 'v1_fallback_used', 'v3_used'))):
                raise ValueError('Polar reproduction requires the frozen pure-V2 availability/status columns and C1/C2 assets; supply the portable V2 manifest, not guessed geometry')
            if str(row['geometry_version']) not in ('v2', '2') or data_explicit_bool(row['v1_fallback_used'], 'v1_fallback_used') or data_explicit_bool(row['v3_used'], 'v3_used'):
                raise ValueError('Only pure V2 is allowed; no V1/hybrid/V3 substitutions')
            if row['c1_status'] != 'ok' or row['c2_status'] != 'ok':
                unavailable.append({'image_id': row['image_id'], 'reason': row.get('failure_reason', 'V2 common availability')})
                continue
            key = 'c1' if condition == 'E1-C1' else 'c2'
            if row.get(key + '_path') is None or row.get(key + '_validity_path') is None:
                raise FileNotFoundError(f"Missing frozen {key.upper()} texture/validity for {row['image_id']}. Supply the original cache or explicitly rebuild using the frozen V2 method and audited availability.")
        selected.append(dict(row))
    if not selected:
        raise ValueError(f'No eligible {split} images for {condition}')
    return (selected, unavailable)

def data_pk_eligible(records):
    sources = defaultdict(set)
    for row in records:
        if row['split'] != 'train':
            raise ValueError('PK eligibility must use training records only')
        sources[row['iris_id']].add(row['source_id'])
    return [r for r in records if len(sources[r['iris_id']]) >= 2]

class data_IrisDataset(Dataset):
    """Exact cached/native tensors with explicit split access and no resizing of caches."""

    def __init__(self, records, config, *, split, augment=False):
        self.records = list(records)
        self.condition = config['condition']
        self.normalization = dict(config['normalization'])
        self.evaluation = not augment
        if not self.records or any((r['split'] != split for r in self.records)):
            raise ValueError('Dataset access requires one explicit, matching partition')
        if augment and split != 'train':
            raise ValueError('Augmentation is permitted on training captures only')
        aug = config['augmentation']
        self.augmentation = {k: aug[k] for k in ('probability', 'brightness', 'contrast')} if augment and aug['enabled'] else None
        self.dataset_labels = {d: i for i, d in enumerate((*config_DOMAINS, *config_EXTERNAL_DOMAINS))}
        self.subject_labels = {k: i for i, k in enumerate(sorted({(r['dataset'], r['subject_id']) for r in self.records}))}
        self.iris_labels = {k: i for i, k in enumerate(sorted({r['iris_id'] for r in self.records}))}
        self.source_labels = {k: i for i, k in enumerate(sorted({r['source_id'] for r in self.records}))}

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        support = validity = None
        native_shape = (int(row['height']), int(row['width'])) if data_present(row.get('height')) and data_present(row.get('width')) else None
        if self.condition.startswith('E1-C'):
            key = 'c1' if self.condition == 'E1-C1' else 'c2'
            pixels = preprocessing_read_gray(row[key + '_path'])
            validity = preprocessing_read_binary(row[key + '_validity_path'], (64, 512))
        else:
            if row.get('raw_path') is not None:
                pixels = preprocessing_read_gray(row['raw_path'])
                if pixels.shape != (320, 320):
                    raise ValueError('Cached raw pixels must already be 320x320; no extra resize is allowed')
            else:
                if row.get('native_path') is None:
                    raise FileNotFoundError('Provide raw_path or native_path in the image manifest')
                native = preprocessing_read_gray(row['native_path'], native=True)
                if native_shape is not None and native.shape != native_shape:
                    raise ValueError('Native image dimensions disagree with the authoritative inventory')
                native_shape = native.shape
                pixels = preprocessing_resize_raw(native, preprocessing_geometry(native.shape[1], native.shape[0]))
            if self.condition in ('E1-B1', 'E1-B2'):
                column = 'b1_support_path' if self.condition == 'E1-B1' else 'b2_support_path'
                if row.get(column) is not None:
                    support = preprocessing_read_binary(row[column], (320, 320))
                else:
                    if any((row.get(k + '_mask_path') is None for k in ('iris', 'pupil', 'eyelashes'))):
                        raise FileNotFoundError('Provide the frozen support cache or the native iris/pupil/eyelash masks; no segmentation model is substituted')
                    masks = {k: preprocessing_read_binary(row[k + '_mask_path']) for k in ('iris', 'pupil', 'eyelashes')}
                    if len({m.shape for m in masks.values()}) != 1:
                        raise ValueError('Native segmentation masks do not align')
                    if native_shape is None:
                        if row.get('native_path') is None:
                            raise ValueError('Native-mask fallback needs original width/height or native_path to verify alignment with the raw frame')
                        native_shape = preprocessing_read_gray(row['native_path'], native=True).shape
                    if masks['iris'].shape != native_shape:
                        raise ValueError('Native mask/image geometry mismatch; do not resize mismatched anatomy into apparent alignment')
                    native_support = masks['iris'] & ~masks['pupil']
                    if self.condition == 'E1-B2':
                        native_support &= ~masks['eyelashes']
                    support = preprocessing_transform_support(native_support, preprocessing_geometry(native_support.shape[1], native_support.shape[0]))
        image = preprocessing_tensor_from_pixels(pixels, self.condition, self.normalization, support=support, validity=validity, augmentation=self.augmentation)
        scalar = lambda x: torch.tensor(x, dtype=torch.long)
        return {'image': image, 'dataset_id': scalar(self.dataset_labels[row['dataset']]), 'subject_label': scalar(self.subject_labels[row['dataset'], row['subject_id']]), 'eye_label': scalar(0 if row['eye_side'] == 'L' else 1), 'iris_label': scalar(self.iris_labels[row['iris_id']]), 'source_label': scalar(self.source_labels[row['source_id']]), **{k: row[k] for k in ('image_id', 'iris_id', 'dataset', 'subject_id', 'eye_side', 'split')}}
data_FROZEN_MEMBERSHIP = {'seed': 42, 'datasets': {'CASIA-Interval': {'test': ['009', '010', '018', '023', '025', '028', '039', '044', '057', '070', '074', '096', '099', '101', '114', '125', '130', '131', '141', '149', '152', '153', '165', '175', '176', '182', '188', '191', '204', '205', '212', '218', '219', '231', '233', '234', '249'], 'train': ['001', '003', '004', '006', '007', '008', '011', '012', '013', '014', '015', '016', '019', '020', '021', '024', '026', '030', '032', '033', '034', '036', '037', '038', '040', '041', '042', '043', '045', '046', '047', '048', '051', '052', '053', '054', '055', '058', '059', '060', '061', '062', '063', '064', '065', '066', '067', '068', '069', '071', '072', '073', '075', '076', '077', '078', '080', '081', '083', '084', '085', '086', '087', '089', '090', '092', '093', '095', '097', '102', '105', '106', '107', '108', '110', '111', '112', '113', '116', '117', '118', '119', '120', '121', '123', '124', '126', '127', '128', '129', '132', '133', '134', '135', '136', '137', '138', '139', '140', '142', '143', '144', '146', '147', '148', '150', '151', '154', '155', '156', '157', '158', '159', '160', '161', '163', '164', '166', '167', '168', '169', '170', '171', '172', '173', '178', '179', '181', '184', '185', '186', '187', '189', '190', '192', '193', '194', '195', '196', '197', '198', '200', '201', '202', '203', '206', '207', '209', '210', '211', '213', '214', '215', '216', '222', '223', '224', '225', '226', '227', '228', '229', '230', '232', '236', '237', '240', '241', '242', '243', '244', '245', '246', '248'], 'validation': ['002', '005', '017', '022', '027', '029', '031', '035', '049', '050', '056', '079', '082', '088', '091', '094', '098', '100', '103', '104', '109', '115', '122', '145', '162', '174', '177', '180', '183', '199', '208', '217', '220', '221', '235', '238', '239', '247']}, 'CUST-Iris': {'test': ['11', '17', '23', '28', '109', '114', '116', '120', '141', '142', '143', '147', '158', '173', '174', '176', '200', '216', '224', '226', '228', '244', '247', '248', '251', '252', '259', '264', '272', '279', '286', '295', '296', '302', '309', '313', '314', '320', '334', '355', '356', '359', '364', '365', '372', '376', '379', '382', '387', '393', '395', '413', '417', '423'], 'train': ['1', '2', '10', '12', '13', '14', '15', '16', '19', '21', '22', '24', '25', '27', '29', '30', '31', '32', '33', '34', '35', '37', '38', '39', '40', '41', '100', '102', '103', '104', '106', '108', '110', '111', '112', '113', '115', '118', '119', '121', '123', '124', '125', '126', '128', '129', '131', '132', '133', '134', '136', '137', '138', '139', '144', '145', '146', '148', '149', '150', '151', '154', '155', '156', '157', '159', '160', '161', '162', '163', '164', '165', '166', '167', '169', '170', '171', '172', '175', '177', '178', '179', '180', '181', '182', '183', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195', '196', '197', '198', '201', '202', '203', '205', '206', '207', '209', '210', '211', '214', '215', '217', '218', '219', '220', '221', '222', '223', '225', '227', '229', '230', '232', '233', '234', '235', '236', '238', '239', '241', '242', '243', '245', '246', '250', '254', '255', '256', '257', '258', '260', '262', '263', '265', '266', '267', '269', '271', '273', '275', '276', '277', '278', '280', '281', '282', '283', '284', '285', '287', '288', '289', '290', '291', '292', '294', '297', '298', '299', '300', '301', '304', '307', '308', '310', '311', '312', '316', '317', '318', '319', '322', '323', '324', '325', '326', '327', '328', '329', '330', '331', '332', '333', '336', '337', '339', '340', '341', '342', '343', '344', '345', '347', '348', '349', '352', '354', '360', '361', '362', '363', '366', '367', '368', '369', '370', '371', '373', '374', '375', '377', '378', '380', '381', '383', '386', '388', '389', '391', '392', '394', '397', '399', '401', '402', '403', '404', '405', '406', '407', '408', '409', '410', '411', '412', '415', '416', '418', '419', '420', '421', '422', '424'], 'validation': ['3', '18', '20', '26', '36', '101', '105', '107', '117', '122', '127', '130', '135', '140', '152', '153', '168', '184', '185', '199', '204', '208', '212', '213', '231', '237', '240', '249', '253', '261', '268', '270', '274', '293', '303', '305', '306', '315', '321', '335', '338', '346', '350', '351', '353', '357', '358', '384', '385', '390', '396', '398', '400', '414']}, 'IITD': {'test': ['004', '013', '021', '026', '033', '034', '042', '059', '061', '064', '070', '072', '074', '081', '088', '092', '115', '117', '118', '128', '135', '153', '158', '166', '169', '174', '178', '189', '192', '198', '206', '214'], 'train': ['002', '003', '005', '007', '008', '009', '010', '011', '012', '014', '015', '016', '017', '018', '019', '020', '023', '024', '025', '027', '029', '030', '032', '035', '036', '037', '038', '039', '040', '041', '043', '044', '045', '046', '047', '048', '049', '050', '051', '052', '053', '055', '057', '060', '062', '063', '065', '066', '067', '068', '069', '076', '077', '078', '079', '080', '082', '083', '084', '085', '086', '091', '094', '095', '096', '097', '098', '099', '100', '101', '102', '103', '105', '106', '107', '108', '109', '112', '113', '114', '116', '119', '120', '121', '122', '123', '126', '127', '129', '130', '131', '132', '133', '137', '138', '139', '140', '142', '143', '144', '145', '146', '147', '148', '149', '152', '154', '155', '156', '157', '159', '160', '162', '163', '164', '165', '167', '168', '170', '171', '172', '175', '176', '179', '181', '182', '183', '184', '185', '186', '187', '188', '190', '191', '195', '196', '197', '199', '200', '201', '202', '203', '204', '207', '208', '209', '210', '211', '212', '213', '215', '217'], 'validation': ['006', '022', '028', '031', '054', '056', '058', '071', '073', '075', '087', '089', '090', '093', '104', '110', '111', '124', '125', '134', '136', '141', '150', '151', '161', '173', '177', '180', '193', '194', '205', '216', '218']}}, 'excluded_image_ids': ['IITD:001_01_L', 'IITD:001_02_L', 'IITD:001_03_L', 'IITD:001_04_L', 'IITD:001_05_L', 'IITD:001_06_L', 'IITD:001_06_R', 'IITD:001_07_L', 'IITD:001_07_R', 'IITD:001_08_L', 'IITD:001_08_R', 'IITD:001_09_L', 'IITD:001_09_R', 'IITD:001_10_L', 'IITD:001_10_R', 'IITD:018_08_R', 'IITD:038_02_L', 'IITD:111_02_L', 'IITD:128_10_R', 'IITD:140_09_R', 'IITD:150_02_L', 'IITD:185_10_R', 'IITD:207_06_L', 'IITD:212_06_L', 'IITD:213_07_L'], 'excluded_subjects': [{'dataset': 'IITD', 'subject_id': '001'}]}
data_INVENTORY_COUNTS = {'raw': {'CASIA-Interval': {'train': 1847, 'validation': 396, 'test': 396}, 'CUST-Iris': {'train': 2016, 'validation': 432, 'test': 432}, 'IITD': {'train': 1509, 'validation': 327, 'test': 319}}, 'polar_common': {'CASIA-Interval': {'train': 1842, 'validation': 394, 'test': 395}, 'CUST-Iris': {'train': 2002, 'validation': 426, 'test': 427}, 'IITD': {'train': 1476, 'test': 308, 'validation': 318}}}

### ResNet-18 shared embedding encoder

In [ ]:
class resnet_IrisEmbeddingEncoder(nn.Module):

    def __init__(self, pretrained: bool=True, pretrained_file=None) -> None:
        super().__init__()
        self.backbone = timm.create_model('resnet18', pretrained=pretrained, num_classes=0, in_chans=1, global_pool='avg', **{'pretrained_cfg_overlay': {'file': str(pretrained_file), 'hf_hub_id': None}} if pretrained_file else {})
        if self.backbone.num_features != 512:
            raise ValueError('Expected the 512-dimensional ResNet18 backbone')
        self.backbone.requires_grad_(True)
        self.projection = nn.Sequential(nn.Linear(512, 512), nn.GELU(), nn.LayerNorm(512), nn.Linear(512, 256))

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if images.ndim != 4 or tuple(images.shape[1:]) != (1, 320, 320):
            raise ValueError(f'Expected [B,1,320,320], got {tuple(images.shape)}')
        projected = self.projection(self.backbone(images))
        with torch.autocast(device_type=images.device.type, enabled=False):
            return F.normalize(projected.float(), p=2, dim=1)

### SwinV2-Tiny shared embedding encoder

In [ ]:
swin_MODEL_NAME = 'swinv2_tiny_window8_256'
swin_PRETRAINED_TAG = 'ms_in1k'
swin_ALLOWED_SHAPES = ((320, 320), (64, 512))

class swin_Float32WindowAttention(WindowAttention):
    """User-approved FP32 attention; preserve timm operations and parameter keys.

    Padded zero tokens produce nonfinite FP16 cosine-attention gradients in the
    installed timm implementation. Disable autocast for this module only. The
    patch embedding, MLPs, other backbone operations and head still use AMP.
    """

    def forward(self, x, mask=None):
        with torch.autocast(device_type=x.device.type, enabled=False):
            return super().forward(x.float(), None if mask is None else mask.float())

def swin_use_float32_attention(model):
    """Change execution only, without initializing tensors or consuming RNG."""
    modules = [module for module in model.modules() if type(module) is WindowAttention]
    if len(modules) != 12:
        raise ValueError(f'Expected 12 native SwinV2 attention modules, got {len(modules)}')
    for module in modules:
        module.__class__ = swin_Float32WindowAttention
    return len(modules)

class swin_SwinBackbone(nn.Module):
    """Expose the first patch convolution to the unchanged E1 gradient probe.

    The property does not register another module or duplicate any parameters.
    Windowing, pooling and pretrained loading stay in timm. Attention uses the
    approved float32 execution context around timm's unchanged calculation.
    """

    def __init__(self, img_size, pretrained=True, pretrained_file=None):
        super().__init__()
        self.model = timm.create_model(swin_MODEL_NAME, pretrained=pretrained, num_classes=0, in_chans=1, img_size=tuple(img_size), global_pool='avg', strict_img_size=False, **{'pretrained_cfg_overlay': {'file': str(pretrained_file), 'hf_hub_id': None}} if pretrained_file else {})
        self.float32_attention_modules = swin_use_float32_attention(self.model)
        self.num_features = int(self.model.num_features)
        if self.num_features != 768:
            raise ValueError(f'Unexpected SwinV2-Tiny features: {self.num_features}')
        if self.model.pretrained_cfg['tag'] != swin_PRETRAINED_TAG:
            raise ValueError('Unexpected pretrained tag')

    @property
    def conv1(self):
        return self.model.patch_embed.proj

    def forward(self, images):
        features = self.model(images)
        if features.ndim != 2 or features.shape != (images.shape[0], self.num_features):
            raise ValueError(f'Expected pooled [B,{self.num_features}], got {tuple(features.shape)}')
        return features

class swin_SwinEmbeddingEncoder(nn.Module):

    def __init__(self, img_size, pretrained=True, pretrained_file=None):
        super().__init__()
        self.img_size = tuple(img_size)
        if self.img_size not in swin_ALLOWED_SHAPES:
            raise ValueError(f'Unapproved E1 input shape: {self.img_size}')
        self.backbone = swin_SwinBackbone(self.img_size, pretrained=pretrained, pretrained_file=pretrained_file)
        self.backbone.requires_grad_(True)
        self.projection = nn.Sequential(nn.Linear(self.backbone.num_features, 512), nn.GELU(), nn.LayerNorm(512), nn.Linear(512, 256))

    def forward(self, images):
        if images.ndim != 4 or tuple(images.shape[1:]) != (1, *self.img_size):
            raise ValueError(f'Expected [B,1,{self.img_size[0]},{self.img_size[1]}], got {tuple(images.shape)}')
        projected = self.projection(self.backbone(images))
        with torch.autocast(device_type=images.device.type, enabled=False):
            return F.normalize(projected.float(), p=2, dim=1)

def swin_describe_windows(encoder):
    return [dict(stage=i, input_resolution=list(stage.input_resolution), output_resolution=list(stage.output_resolution), windows=[list(block.window_size) for block in stage.blocks], shifts=[list(block.shift_size) for block in stage.blocks], dynamic_masks=[bool(block.dynamic_mask) for block in stage.blocks]) for i, stage in enumerate(encoder.backbone.model.layers)]

### Model construction and strict input guards

In [ ]:
class models_PolarEncoder(resnet_IrisEmbeddingEncoder):
    """Original ResNet polar adapter: only the spatial guard differs."""

    def forward(self, images):
        if images.ndim != 4 or tuple(images.shape[1:]) != (1, 64, 512):
            raise ValueError('Expected [B,1,64,512] polar input')
        projected = self.projection(self.backbone(images))
        with torch.autocast(device_type=images.device.type, enabled=False):
            return F.normalize(projected.float(), p=2, dim=1)

def models_build_model(config, *, pretrained=True, pretrained_asset=None, allow_download=False):
    """Initialize independently, using the original timm RGB-to-gray adaptation.

    `pretrained=False` is reserved for clearly labelled software tests or loading
    a complete trained checkpoint, never a fallback after failed asset loading.
    """
    architecture, condition = (config['architecture_label'], config['condition'])
    if pretrained_asset is not None:
        pretrained_asset = Path(pretrained_asset).expanduser().resolve()
        if not pretrained_asset.is_file():
            raise FileNotFoundError(f'Missing pretrained asset: {pretrained_asset}. Download the documented original timm model.safetensors and set PRETRAINED_ASSET.')
    if pretrained and pretrained_asset is None and (not allow_download):
        raise FileNotFoundError('Provide PRETRAINED_ASSET or explicitly set ALLOW_PRETRAINED_DOWNLOAD=True. No random-weight fallback is used.')
    try:
        if architecture == 'ResNet-18':
            cls = models_PolarEncoder if condition.startswith('E1-C') else resnet_IrisEmbeddingEncoder
            model = cls(pretrained=pretrained, pretrained_file=pretrained_asset)
        elif architecture == 'SwinV2-Tiny':
            model = swin_SwinEmbeddingEncoder(config['input_shape'][1:], pretrained=pretrained, pretrained_file=pretrained_asset)
        else:
            raise ValueError(f'Unsupported architecture: {architecture}')
    except (OSError, RuntimeError) as error:
        raise RuntimeError(f'Could not initialize the original {architecture} pretrained source. Check the documented tagged asset and installed timm version. No alternative weights were substituted. {error}') from error
    if not all((p.requires_grad for p in model.parameters())):
        raise RuntimeError('All backbone and projection parameters must remain trainable')
    return model

def models_model_summary(model, config):
    return {'architecture': config['architecture_label'], 'condition': config['condition'], 'input_shape': ['B', *config['input_shape']], 'output_shape': ['B', 256], 'pooled_features': model.backbone.num_features, 'trainable_parameters': sum((p.numel() for p in model.parameters() if p.requires_grad)), 'single_shared_encoder': True, 'l2_normalized_output': True, 'swin_float32_window_attention': config['architecture_label'] == 'SwinV2-Tiny'}

### Complete ROC, EER and low-FAR metric definitions

In [ ]:
metrics_DOMAINS = ('CUST-Iris', 'CASIA-Interval', 'IITD')
metrics_EER_METHOD = 'Linear interpolation on the complete ROC: locate adjacent points where d=FNR-FPR crosses zero; alpha=d0/(d0-d1), EER=FPR0+alpha*(FPR1-FPR0). Tied scores move together. No fixed operating threshold is selected.'
metrics_TAR_METHOD = 'ROC-interpolated TAR on the complete ROC: at an exact repeated FPR select its maximum TPR; between knots interpolate from the last lower-FPR point to the first higher-FPR point, preserving future vertical jumps. These values are not empirical results at a fixed threshold.'

def metrics_scores(values, name: str) -> np.ndarray:
    result = np.asarray(values, dtype=np.float64)
    if result.ndim != 1 or not len(result) or (not np.isfinite(result).all()):
        raise ValueError(f'{name} must be a nonempty finite one-dimensional score array')
    return result

def metrics_interpolated_eer(fpr: np.ndarray, tpr: np.ndarray) -> float:
    difference = 1.0 - tpr - fpr
    right = int(np.flatnonzero(difference <= 0)[0])
    if difference[right] == 0 or right == 0:
        return float(fpr[right])
    left = right - 1
    alpha = difference[left] / (difference[left] - difference[right])
    return float(fpr[left] + alpha * (fpr[right] - fpr[left]))

def metrics_interpolated_tar(fpr: np.ndarray, tpr: np.ndarray, far: float) -> float:
    right = int(np.searchsorted(fpr, far, side='right'))
    left = right - 1
    if right == len(fpr):
        return float(tpr[-1])
    if left < 0:
        return float(tpr[0])
    if fpr[left] == far:
        return float(tpr[left])
    alpha = (far - fpr[left]) / (fpr[right] - fpr[left])
    return float(tpr[left] + alpha * (tpr[right] - tpr[left]))

def metrics_verification_metrics(genuine_scores, impostor_scores) -> dict:
    """Return float64 metrics from scores for both classes, without tuning.

    AUC is the trapezoidal area of the full ROC. EER is obtained by interpolating
    the FNR-FPR zero crossing. FAR resolution is one false acceptance divided by
    the number of impostor comparisons; interpolation does not improve that
    empirical resolution or establish independent-trial confidence intervals.
    """
    genuine = metrics_scores(genuine_scores, 'Genuine scores')
    impostor = metrics_scores(impostor_scores, 'Impostor scores')
    labels = np.concatenate((np.ones(len(genuine), dtype=np.uint8), np.zeros(len(impostor), dtype=np.uint8)))
    scores = np.concatenate((genuine, impostor))
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1, drop_intermediate=False)
    resolution = 1.0 / len(impostor)
    notes = []
    for far in (0.01, 0.001):
        if far < resolution:
            notes.append(f'FAR={far:g} is below one observed false acceptance ({resolution:.12g}); its TAR is an ROC interpolation, not an empirically resolved fixed-threshold operating point.')
    return {'roc_auc': float(auc(fpr, tpr)), 'eer': metrics_interpolated_eer(fpr, tpr), 'tar_at_far_1pct': metrics_interpolated_tar(fpr, tpr, 0.01), 'tar_at_far_0_1pct': metrics_interpolated_tar(fpr, tpr, 0.001), 'num_genuine': int(len(genuine)), 'num_impostor': int(len(impostor)), 'far_resolution': resolution, 'eer_method': metrics_EER_METHOD, 'tar_method': metrics_TAR_METHOD, 'tar_is_roc_interpolated': True, 'low_far_notes': notes}

def metrics_numpy_embedding(value) -> np.ndarray:
    if hasattr(value, 'detach'):
        value = value.detach().cpu().numpy()
    result = np.asarray(value, dtype=np.float32)
    if result.ndim != 1 or not len(result) or (not np.isfinite(result).all()):
        raise ValueError('Each embedding must be a finite nonempty vector')
    return result

def metrics_pair_chunks(pairs, chunk_size: int):
    if hasattr(pairs, 'to_batches'):
        for batch in pairs.to_batches(max_chunksize=chunk_size):
            yield batch.to_pylist()
    elif isinstance(pairs, Sequence) and (not isinstance(pairs, (str, bytes))):
        for offset in range(0, len(pairs), chunk_size):
            yield pairs[offset:offset + chunk_size]
    else:
        raise TypeError('Pairs must be a sequence of dictionaries or a PyArrow Table')

def metrics_evaluate_validation(embeddings_by_id: Mapping, pairs_by_dataset: Mapping, chunk_size: int=65536) -> dict:
    """Score every frozen pair and macro-average the three validation EERs.

    No images are opened here. Embeddings are CPU float32 vectors, converted
    from tensors if needed and normalized before cosine dot products. Scoring
    is chunked without choosing, discarding or sampling pairs based on scores.
    Pair membership/subject/source eligibility is sealed by the input protocol;
    this function additionally rejects self-comparisons and domain mismatches.
    """
    if not isinstance(chunk_size, Integral) or chunk_size <= 0:
        raise ValueError('chunk_size must be a positive integer')
    if set(pairs_by_dataset) != set(metrics_DOMAINS):
        raise ValueError('Macro-validation requires exactly CUST-Iris, CASIA-Interval and IITD')
    if not embeddings_by_id:
        raise ValueError('Validation embeddings cannot be empty')
    ids = sorted(embeddings_by_id)
    vectors = [metrics_numpy_embedding(embeddings_by_id[key]) for key in ids]
    if len({v.shape for v in vectors}) != 1:
        raise ValueError('Validation embeddings must share one feature dimension')
    matrix = np.stack(vectors)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if not np.isfinite(norms).all() or (norms <= 0).any():
        raise ValueError('Validation embeddings must have finite positive norms')
    matrix = matrix / norms
    index = {key: i for i, key in enumerate(ids)}
    per_dataset = {}
    for domain in metrics_DOMAINS:
        genuine_parts, impostor_parts = ([], [])
        for rows in metrics_pair_chunks(pairs_by_dataset[domain], int(chunk_size)):
            left, right, genuine = ([], [], [])
            for row in rows:
                if row['dataset'] != domain:
                    raise ValueError('Pair dataset differs from its frozen domain group')
                a, b = (row['image_id_a'], row['image_id_b'])
                if a == b:
                    raise ValueError('Self/identical-image comparisons are forbidden')
                if a not in index or b not in index:
                    raise ValueError(f'Frozen validation pair lacks an embedding: {a}, {b}')
                if not isinstance(row['is_genuine'], (bool, np.bool_)):
                    raise ValueError('Pair is_genuine must be an explicit boolean')
                left.append(index[a])
                right.append(index[b])
                genuine.append(row['is_genuine'])
            if not left:
                continue
            similarities = np.einsum('ij,ij->i', matrix[left], matrix[right], dtype=np.float32)
            similarities = np.clip(similarities, -1.0, 1.0)
            mask = np.asarray(genuine, dtype=bool)
            genuine_parts.append(similarities[mask])
            impostor_parts.append(similarities[~mask])
        genuine_scores = np.concatenate(genuine_parts) if genuine_parts else np.empty(0)
        impostor_scores = np.concatenate(impostor_parts) if impostor_parts else np.empty(0)
        per_dataset[domain] = metrics_verification_metrics(genuine_scores, impostor_scores)
    return {'per_dataset': per_dataset, 'macro_eer': float(np.mean([per_dataset[d]['eer'] for d in metrics_DOMAINS], dtype=np.float64)), 'score': 'cosine similarity of L2-normalized float32 embeddings', 'selection_metric': 'unweighted macro-average of three per-dataset validation EERs'}

### Physical-eye verification-pair checks

In [ ]:
pairs_KINDS = ('genuine', 'impostor', 'opposite_eye_diagnostic')

def pairs_validate_pairs(rows, records, *, domain, split, allow_availability_subset=False):
    by_id = {r['image_id']: r for r in records}
    pairs, unavailable, seen = ([], [], set())
    for raw in rows:
        row = dict(raw)
        a, b = (str(row['image_id_a']), str(row['image_id_b']))
        if a not in by_id or b not in by_id:
            if allow_availability_subset:
                unavailable.append((a, b))
                continue
            raise ValueError(f'Frozen pair lacks a member in its declared population: {a}, {b}')
        left, right = (by_id[a], by_id[b])
        if left['split'] != split or right['split'] != split or left['dataset'] != domain or (right['dataset'] != domain):
            raise ValueError('Pair crosses a dataset or frozen partition boundary')
        if a == b or left['source_id'] == right['source_id']:
            raise ValueError('Self/identical-source images may not count as verification pairs')
        kind = row.get('pair_type')
        if not kind:
            kind = 'genuine' if data_explicit_bool(row['is_genuine'], 'is_genuine') else 'impostor'
        if kind not in pairs_KINDS:
            raise ValueError(f'Unexpected pair category: {kind}')
        genuine = left['iris_id'] == right['iris_id']
        impostor = left['subject_id'] != right['subject_id']
        opposite = left['subject_id'] == right['subject_id'] and left['eye_side'] != right['eye_side']
        if not {'genuine': genuine, 'impostor': impostor, 'opposite_eye_diagnostic': opposite}[kind]:
            raise ValueError(f'Pair labels disagree with dataset+subject+eye: {a}, {b}')
        if 'is_genuine' in row and data_explicit_bool(row['is_genuine'], 'is_genuine') != genuine:
            raise ValueError("Pair's is_genuine contradicts physical-eye identity")
        if 'label' in row and int(row['label']) != int(genuine):
            raise ValueError("Pair's label contradicts physical-eye identity")
        if 'dataset' in row and row['dataset'] != domain:
            raise ValueError("Pair's dataset disagrees with its members")
        if 'split' in row and row['split'] != split:
            raise ValueError("Pair's split disagrees with its members")
        if 'primary_evaluation' in row and data_explicit_bool(row['primary_evaluation'], 'primary_evaluation') != (kind != 'opposite_eye_diagnostic'):
            raise ValueError("Pair's primary/diagnostic flag contradicts its category")
        for side, member in (('a', left), ('b', right)):
            expected_fields = {f'iris_id_{side}': member['iris_id'], f'eye_side_{side}': member['eye_side'], f'partition_key_{side}': f"{domain}:{member['subject_id']}"}
            for key, expected in expected_fields.items():
                if key in row and row[key] != expected:
                    raise ValueError(f"Frozen pair's {key} contradicts the authoritative image label")
        key = tuple(sorted((a, b)))
        if key in seen:
            raise ValueError('Repeated/reversed pair in a frozen population')
        seen.add(key)
        pairs.append({**row, 'image_id_a': a, 'image_id_b': b, 'dataset': domain, 'split': split, 'pair_type': kind, 'is_genuine': bool(genuine), 'primary_evaluation': kind != 'opposite_eye_diagnostic'})
    return (pairs, unavailable)

def pairs_load_validation_pairs(files, records, condition, *, original_validation_records=None):
    if set(files) != set(config_DOMAINS):
        raise ValueError('The three original validation datasets and fixed pair files are required')
    result = {}
    for domain, path in files.items():
        original = original_validation_records if original_validation_records is not None else records
        if condition.startswith('E1-C') and original_validation_records is None:
            raise ValueError('Supply the complete frozen validation inventory before taking its pure-V2 availability view')
        pairs, _ = pairs_validate_pairs(data_load_table(path).to_dict('records'), original, domain=domain, split='validation')
        allowed = {r['image_id'] for r in records}
        if condition.startswith('E1-C'):
            pairs = [p for p in pairs if p['image_id_a'] in allowed and p['image_id_b'] in allowed]
        elif any((p['image_id_a'] not in allowed or p['image_id_b'] not in allowed for p in pairs)):
            raise ValueError('Cartesian validation pair population differs from retained validation membership')
        if not any((p['is_genuine'] for p in pairs)) or not any((p['pair_type'] == 'impostor' for p in pairs)):
            raise ValueError(f'Insufficient frozen validation pairs for {domain}')
        result[domain] = pairs
    return result

### Explicit masked triplet and supervised contrastive objectives

In [ ]:
@dataclass(frozen=True)
class losses_PairMasks:
    positive: torch.Tensor
    negative: torch.Tensor
    allowed: torch.Tensor

def losses_build_pair_masks(dataset_ids, subject_ids, eye_ids, source_ids) -> losses_PairMasks:
    labels = (dataset_ids, subject_ids, eye_ids, source_ids)
    if any((x.ndim != 1 for x in labels)) or len({len(x) for x in labels}) != 1:
        raise ValueError('Labels must be equally sized one-dimensional tensors')
    if len({x.device for x in labels}) != 1:
        raise ValueError('All labels must share a device')
    n = len(dataset_ids)
    same_domain = dataset_ids[:, None].eq(dataset_ids[None, :])
    same_subject = subject_ids[:, None].eq(subject_ids[None, :])
    same_eye = eye_ids[:, None].eq(eye_ids[None, :])
    distinct_source = source_ids[:, None].ne(source_ids[None, :])
    nonself = ~torch.eye(n, dtype=torch.bool, device=dataset_ids.device)
    eligible = same_domain & distinct_source & nonself
    positive = eligible & same_subject & same_eye
    negative = eligible & ~same_subject
    return losses_PairMasks(positive, negative, positive | negative)

def losses_validate(embeddings: torch.Tensor, masks: losses_PairMasks) -> None:
    if embeddings.ndim != 2 or not len(embeddings):
        raise ValueError('Embeddings must be a nonempty [B,D] tensor')
    n = embeddings.shape[0]
    for mask in (masks.positive, masks.negative, masks.allowed):
        if mask.shape != (n, n) or mask.dtype != torch.bool or mask.device != embeddings.device:
            raise ValueError('Pair masks must be boolean [B,B] on the embedding device')
        if mask.diagonal().any() or not torch.equal(mask, mask.T):
            raise ValueError('Pair masks must be symmetric and exclude self-pairs')
    if (masks.positive & masks.negative).any() or not torch.equal(masks.allowed, masks.positive | masks.negative):
        raise ValueError('Allowed pairs must be exactly disjoint positives and negatives')
    if not masks.positive.any(dim=1).all():
        raise ValueError('Each anchor requires at least one distinct-source positive')
    if not masks.negative.any(dim=1).all():
        raise ValueError('Each anchor requires at least one same-domain different-subject negative')

def losses_normalized_float(embeddings: torch.Tensor) -> torch.Tensor:
    z = embeddings.float()
    if z.ndim != 2 or not torch.isfinite(z).all():
        raise ValueError('Embeddings must be finite [B,D]')
    scale = z.abs().amax(dim=1, keepdim=True)
    if (scale == 0).any():
        raise ValueError('Zero embeddings do not define cosine similarity')
    return F.normalize(z / scale, p=2, dim=1)

def losses_pairwise_squared_distances(embeddings: torch.Tensor) -> torch.Tensor:
    with torch.autocast(device_type=embeddings.device.type, enabled=False):
        z = losses_normalized_float(embeddings)
        return (2.0 - 2.0 * (z @ z.T)).clamp_min(0.0)

def losses_reduce(values: torch.Tensor, reduction: str) -> torch.Tensor:
    if reduction == 'none':
        return values
    if reduction == 'mean':
        return values.mean()
    raise ValueError("Reduction must be 'mean' or 'none'")

def losses_batch_hard_triplet_loss(embeddings, masks: losses_PairMasks, margin=0.3, reduction='mean'):
    if margin < 0:
        raise ValueError('Margin must be nonnegative')
    losses_validate(embeddings, masks)
    with torch.autocast(device_type=embeddings.device.type, enabled=False):
        distance = losses_pairwise_squared_distances(embeddings)
        hardest_positive = distance.masked_fill(~masks.positive, -torch.inf).amax(dim=1)
        hardest_negative = distance.masked_fill(~masks.negative, torch.inf).amin(dim=1)
        return losses_reduce(F.relu(hardest_positive - hardest_negative + margin), reduction)

def losses_supervised_contrastive_loss(embeddings, masks: losses_PairMasks, temperature=0.07, reduction='mean'):
    if temperature <= 0:
        raise ValueError('Temperature must be positive')
    losses_validate(embeddings, masks)
    with torch.autocast(device_type=embeddings.device.type, enabled=False):
        z = losses_normalized_float(embeddings)
        logits = z @ z.T / temperature
        log_denominator = torch.logsumexp(logits.masked_fill(~masks.allowed, -torch.inf), dim=1)
        log_probability = logits - log_denominator[:, None]
        positive_log_probability = torch.where(masks.positive, log_probability, 0.0)
        losses = -positive_log_probability.sum(dim=1) / masks.positive.sum(dim=1)
        return losses_reduce(losses, reduction)

def losses_metric_objective(embeddings, dataset_ids, subject_ids, eye_ids, source_ids, margin=0.3, temperature=0.07, supcon_weight=0.5):
    if supcon_weight < 0:
        raise ValueError('SupCon weight must be nonnegative')
    masks = losses_build_pair_masks(dataset_ids, subject_ids, eye_ids, source_ids)
    triplet = losses_batch_hard_triplet_loss(embeddings, masks, margin)
    supcon = losses_supervised_contrastive_loss(embeddings, masks, temperature)
    return {'total': triplet + supcon_weight * supcon, 'triplet': triplet, 'supcon': supcon}

### Domain-balanced PK sampler

In [ ]:
class sampler_DomainBalancedPKSampler(Sampler[list[int]]):
    """Default P=12,K=2: four distinct subjects/eyes per each of three domains.

    Each batch selects subjects independently without replacement within domain,
    chooses one eligible eye per subject, then K distinct canonical sources.
    Batches sample with replacement across the epoch: this intentionally balances
    domains of unequal size and does not promise one visit to every capture.
    Default epoch length is ceil(all training records /24)=224 for this inventory.
    `set_epoch` changes a private deterministic RNG stream; global RNG is untouched.
    """

    def __init__(self, records: list[dict], seed: int=42, batches_per_epoch: int | None=None, identities_per_domain: int=4, captures_per_identity: int=2):
        if not records:
            raise ValueError('Training records cannot be empty')
        if not isinstance(identities_per_domain, Integral) or identities_per_domain < 1:
            raise ValueError('identities_per_domain must be positive')
        if not isinstance(captures_per_identity, Integral) or captures_per_identity < 2:
            raise ValueError('K must be at least2 to provide an anchor positive')
        self.records = list(records)
        self.seed = int(seed)
        self.epoch = 0
        self.identities_per_domain = int(identities_per_domain)
        self.captures_per_identity = int(captures_per_identity)
        self.batch_size = len(config_DOMAINS) * self.identities_per_domain * self.captures_per_identity
        if batches_per_epoch is None:
            batches_per_epoch = math.ceil(len(records) / self.batch_size)
        if not isinstance(batches_per_epoch, Integral) or batches_per_epoch < 1:
            raise ValueError('batches_per_epoch must be positive')
        self.batches_per_epoch = int(batches_per_epoch)
        all_irises = {domain: defaultdict(lambda: defaultdict(list)) for domain in config_DOMAINS}
        image_ids = set()
        for index, row in enumerate(records):
            if row.get('split', 'train') != 'train':
                raise ValueError('PK sampler accepts frozen training records only')
            domain = row['dataset']
            if domain not in all_irises:
                raise ValueError(f'Unexpected domain: {domain}')
            if row['iris_id'] != f"{domain}:{row['subject_id']}:{row['eye_side']}":
                raise ValueError('Sampler requires dataset+subject+eye iris identities')
            source = row.get('source_id')
            if not isinstance(source, str) or not source:
                raise ValueError('Canonical source_id is required; no filename fallback')
            if row['image_id'] in image_ids:
                raise ValueError('Repeated image IDs in sampler records')
            image_ids.add(row['image_id'])
            all_irises[domain][row['subject_id'], row['eye_side']][source].append(index)
        self._subjects = {}
        self.eligibility_summary = {}
        for domain in config_DOMAINS:
            eligible = defaultdict(list)
            excluded = []
            for (subject, eye), sources in sorted(all_irises[domain].items()):
                if len(sources) < self.captures_per_identity:
                    excluded.append(f'{domain}:{subject}:{eye}')
                    continue
                normalized = {source: sorted(indices, key=lambda i: records[i]['image_id']) for source, indices in sorted(sources.items())}
                eligible[subject].append((eye, normalized))
            if len(eligible) < self.identities_per_domain:
                raise ValueError(f'{domain} has only{len(eligible)} eligible distinct subjects; requires{self.identities_per_domain}')
            self._subjects[domain] = dict(sorted(eligible.items()))
            self.eligibility_summary[domain] = {'training_records': sum((row['dataset'] == domain for row in records)), 'all_iris_ids': len(all_irises[domain]), 'eligible_iris_ids': sum((len(eyes) for eyes in eligible.values())), 'eligible_subjects': len(eligible), 'ineligible_iris_ids': excluded, 'samples_per_epoch': self.batches_per_epoch * self.identities_per_domain * self.captures_per_identity}

    def __len__(self) -> int:
        return self.batches_per_epoch

    def set_epoch(self, epoch: int) -> None:
        if not isinstance(epoch, Integral) or epoch < 0:
            raise ValueError('Epoch must be a nonnegative integer')
        self.epoch = int(epoch)

    def __iter__(self):
        encoded = f'E1-domain-PK|{self.seed}|epoch:{self.epoch}'.encode('utf-8')
        rng = random.Random(int.from_bytes(hashlib.sha256(encoded).digest(), 'big'))
        for _ in range(self.batches_per_epoch):
            batch = []
            for domain in config_DOMAINS:
                subjects = self._subjects[domain]
                for subject in rng.sample(list(subjects), self.identities_per_domain):
                    _eye, sources = rng.choice(subjects[subject])
                    for source in rng.sample(list(sources), self.captures_per_identity):
                        batch.append(rng.choice(sources[source]))
            rng.shuffle(batch)
            yield batch

### RNG, AdamW and AMP defaults

In [ ]:
def runtime_seed_everything(seed: int=42, deterministic: bool=True) -> None:
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)

def runtime_build_optimizer(model, cfg: dict) -> torch.optim.AdamW:
    parameters = list(model.parameters())
    if not all((p.requires_grad for p in parameters)):
        raise ValueError('The whole encoder, including the backbone, must be trainable')
    return torch.optim.AdamW(parameters, lr=cfg['lr'], weight_decay=cfg['weight_decay'], betas=tuple(cfg['betas']))

def runtime_make_scaler(device: torch.device, amp: bool=True, init_scale: float=1024.0):
    return torch.amp.GradScaler('cuda', init_scale=init_scale, enabled=bool(amp and device.type == 'cuda'))

### Early stopping and RNG state

In [ ]:
@dataclass
class state_EarlyStopping:
    """Select a score only when the reduction strictly exceeds tolerance.

    Epochs are contiguous and 1-based. Qualifying improvements are saved from
    epoch1 onward. Nonimprovements increment the consecutive patience counter
    even before ``min_epochs``; stopping is gated by both minimum epoch and
    patience. Thus a best score at epoch1 followed by no improvements permits
    stopping at epoch30 with the default settings, while a new best at epoch29
    resets the counter and first permits stopping at epoch54. Tiny reductions
    do not move the reference score, so several can eventually exceed tolerance.
    The comparison uses decimal representations of the finite float scores so
    binary subtraction noise cannot turn a reduction of exactly1e-4 into an
    improvement. It introduces no second numerical tolerance.
    """
    min_epochs: int = 30
    patience: int = 25
    tolerance: float = 0.0001
    best_epoch: int = 0
    best_score: float = math.inf
    bad_epochs: int = 0
    last_epoch: int = 0

    def __post_init__(self) -> None:
        if not isinstance(self.min_epochs, int) or isinstance(self.min_epochs, bool) or (not isinstance(self.patience, int)) or isinstance(self.patience, bool) or (self.min_epochs < 1) or (self.patience < 1):
            raise ValueError('Minimum epochs and patience must be positive integers')
        if not math.isfinite(self.tolerance) or self.tolerance < 0:
            raise ValueError('Improvement tolerance must be finite and nonnegative')

    def update(self, epoch: int, score: float) -> dict[str, bool]:
        if not isinstance(epoch, int) or isinstance(epoch, bool) or epoch != self.last_epoch + 1:
            raise ValueError('Early-stopping epochs must be contiguous and 1-based')
        score = float(score)
        if not math.isfinite(score) or not 0 <= score <= 1:
            raise ValueError('Macro-validation EER must be finite and within[0,1]')
        improved = math.isinf(self.best_score) or Decimal(str(self.best_score)) - Decimal(str(score)) > Decimal(str(self.tolerance))
        self.last_epoch = epoch
        if improved:
            self.best_score, self.best_epoch, self.bad_epochs = (score, epoch, 0)
        else:
            self.bad_epochs += 1
        return {'improved': improved, 'should_stop': epoch >= self.min_epochs and self.bad_epochs >= self.patience}

    def state_dict(self) -> dict[str, Any]:
        return {'version': 1, 'min_epochs': self.min_epochs, 'patience': self.patience, 'tolerance': self.tolerance, 'best_epoch': self.best_epoch, 'best_score': self.best_score, 'bad_epochs': self.bad_epochs, 'last_epoch': self.last_epoch}

    def load_state_dict(self, state: Mapping[str, Any]) -> None:
        expected = {'version': 1, 'min_epochs': self.min_epochs, 'patience': self.patience, 'tolerance': self.tolerance}
        if any((state.get(key) != value for key, value in expected.items())):
            raise ValueError('Early-stopping checkpoint settings differ from this run')
        best_epoch, last_epoch, bad_epochs = (state[key] for key in ('best_epoch', 'last_epoch', 'bad_epochs'))
        if any((not isinstance(value, int) or isinstance(value, bool) or value < 0 for value in (best_epoch, last_epoch, bad_epochs))):
            raise ValueError('Invalid early-stopping epoch/counter state')
        best_score = float(state['best_score'])
        if last_epoch == 0:
            valid = best_epoch == 0 and bad_epochs == 0 and (best_score == math.inf)
        else:
            valid = 1 <= best_epoch <= last_epoch and bad_epochs == last_epoch - best_epoch and math.isfinite(best_score) and (0 <= best_score <= 1)
        if not valid:
            raise ValueError('Inconsistent early-stopping checkpoint state')
        self.best_epoch, self.last_epoch = (best_epoch, last_epoch)
        self.bad_epochs, self.best_score = (bad_epochs, best_score)

def state_capture_rng_state(include_cuda: bool | None=None) -> dict[str, Any]:
    """Capture RNGs without creating a CUDA context for an otherwise CPU-only run."""
    if include_cuda is None:
        include_cuda = torch.cuda.is_available() and torch.cuda.is_initialized()
    if include_cuda and (not torch.cuda.is_available()):
        raise RuntimeError('Cannot capture CUDA RNG state without CUDA')
    return {'python': random.getstate(), 'numpy': np.random.get_state(), 'torch': torch.get_rng_state().clone(), 'cuda': [state.cpu().clone() for state in torch.cuda.get_rng_state_all()] if include_cuda else None}

def state_validate_cuda_rng(state: Mapping[str, Any]) -> None:
    cuda_states = state.get('cuda')
    if cuda_states is not None:
        if not torch.cuda.is_available() or len(cuda_states) != torch.cuda.device_count():
            raise ValueError('Saved CUDA RNG device count is unavailable or differs from this machine')

def state_restore_rng_state(state: Mapping[str, Any]) -> None:
    """Restore after constructing/loading all objects, just before the next epoch."""
    state_validate_cuda_rng(state)
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if state.get('cuda') is not None:
        torch.cuda.set_rng_state_all([value.cpu() for value in state['cuda']])

### Full shared-batch gradient steps

In [ ]:
def steps_train_cartesian(model, batch, optimizer, scaler, device, config) -> dict:
    """One shared B24 metric step. AMP overflow skips/logs an update, never changes PK."""
    if tuple(batch['image'].shape) != (24, 1, 320, 320):
        raise ValueError('E1-A requires the complete batch24, without gradient accumulation')
    images = batch['image'].to(device, dtype=torch.float32)
    if not torch.isfinite(images).all():
        raise FloatingPointError('Nonfinite normalized input')
    labels = [batch[key].to(device) for key in ('dataset_id', 'subject_label', 'eye_label', 'source_label')]
    model.train()
    optimizer.zero_grad(set_to_none=True)
    amp = config['runtime']['amp'] and device.type == 'cuda'
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
        embeddings = model(images)
    if not torch.isfinite(embeddings).all():
        raise FloatingPointError('Nonfinite AMP forward embeddings; no preprocessing or precision fallback applied')
    objective = config['objective']
    losses = losses_metric_objective(embeddings, *labels, margin=objective['triplet_margin'], temperature=objective['supcon_temperature'], supcon_weight=objective['supcon_weight'])
    if any((loss.dtype != torch.float32 or not torch.isfinite(loss) for loss in losses.values())):
        raise FloatingPointError('Expected finite float32 metric losses')
    scaler.scale(losses['total']).backward()
    scaler.unscale_(optimizer)
    gradients = [p.grad for p in model.parameters() if p.grad is not None]
    grad_norm = torch.linalg.vector_norm(torch.stack([g.detach().double().norm() for g in gradients]))
    finite = bool(torch.isfinite(grad_norm))
    backbone_gradient = model.backbone.conv1.weight.grad
    projection_gradient = model.projection[-1].weight.grad
    if backbone_gradient is None or projection_gradient is None:
        raise RuntimeError('The backbone and projection must both receive gradients')
    details = {'backbone_gradient_norm': backbone_gradient.detach().float().norm().item(), 'projection_gradient_norm': projection_gradient.detach().float().norm().item()}
    if finite:
        coefficient = (config['optimizer']['gradient_clip_norm'] / (grad_norm + 1e-06)).clamp(max=1.0)
        for gradient in gradients:
            gradient.mul_(coefficient.to(dtype=gradient.dtype))
    elif all((bool(torch.isfinite(gradient).all()) for gradient in gradients)):
        raise FloatingPointError('Aggregate gradient norm overflow before optimizer.step')
    scale_before = scaler.get_scale()
    if scaler.is_enabled():
        scaler.step(optimizer)
        scaler.update()
        skipped = scaler.get_scale() < scale_before
    elif finite:
        optimizer.step()
        skipped = False
    else:
        skipped = True
    if not finite and (not skipped):
        raise RuntimeError('An optimizer update was attempted with nonfinite gradients')
    result = {'total': losses['total'].item(), 'triplet': losses['triplet'].item(), 'supcon': losses['supcon'].item(), 'optimizer_steps': int(not skipped), 'skipped_nonfinite_steps': int(skipped), 'loss_dtype': str(losses['total'].dtype), 'gradient_norm_before_clip': grad_norm.item() if finite else None, 'grad_scaler_scale': scaler.get_scale(), 'amp_enabled': amp, 'embedding_norm_min': embeddings.detach().norm(dim=1).min().item(), 'embedding_norm_max': embeddings.detach().norm(dim=1).max().item(), **{key: value if math.isfinite(value) else None for key, value in details.items()}}
    optimizer.zero_grad(set_to_none=True)
    return result

def steps_train_polar(model, batch, optimizer, scaler, device, config) -> dict:
    """One shared B24 metric step. AMP overflow skips/logs an update, never changes PK."""
    if tuple(batch['image'].shape) != (24, 1, 64, 512):
        raise ValueError('E1-C requires the complete batch24 at 64x512, without gradient accumulation')
    images = batch['image'].to(device, dtype=torch.float32)
    if not torch.isfinite(images).all():
        raise FloatingPointError('Nonfinite normalized input')
    labels = [batch[key].to(device) for key in ('dataset_id', 'subject_label', 'eye_label', 'source_label')]
    model.train()
    optimizer.zero_grad(set_to_none=True)
    amp = config['runtime']['amp'] and device.type == 'cuda'
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
        embeddings = model(images)
    if not torch.isfinite(embeddings).all():
        raise FloatingPointError('Nonfinite AMP forward embeddings; no preprocessing or precision fallback applied')
    objective = config['objective']
    losses = losses_metric_objective(embeddings, *labels, margin=objective['triplet_margin'], temperature=objective['supcon_temperature'], supcon_weight=objective['supcon_weight'])
    if any((loss.dtype != torch.float32 or not torch.isfinite(loss) for loss in losses.values())):
        raise FloatingPointError('Expected finite float32 metric losses')
    scaler.scale(losses['total']).backward()
    scaler.unscale_(optimizer)
    gradients = [p.grad for p in model.parameters() if p.grad is not None]
    grad_norm = torch.linalg.vector_norm(torch.stack([g.detach().double().norm() for g in gradients]))
    finite = bool(torch.isfinite(grad_norm))
    backbone_gradient = model.backbone.conv1.weight.grad
    projection_gradient = model.projection[-1].weight.grad
    if backbone_gradient is None or projection_gradient is None:
        raise RuntimeError('The backbone and projection must both receive gradients')
    details = {'backbone_gradient_norm': backbone_gradient.detach().float().norm().item(), 'projection_gradient_norm': projection_gradient.detach().float().norm().item()}
    if finite:
        coefficient = (config['optimizer']['gradient_clip_norm'] / (grad_norm + 1e-06)).clamp(max=1.0)
        for gradient in gradients:
            gradient.mul_(coefficient.to(dtype=gradient.dtype))
    elif all((bool(torch.isfinite(gradient).all()) for gradient in gradients)):
        raise FloatingPointError('Aggregate gradient norm overflow before optimizer.step')
    scale_before = scaler.get_scale()
    if scaler.is_enabled():
        scaler.step(optimizer)
        scaler.update()
        skipped = scaler.get_scale() < scale_before
    elif finite:
        optimizer.step()
        skipped = False
    else:
        skipped = True
    if not finite and (not skipped):
        raise RuntimeError('An optimizer update was attempted with nonfinite gradients')
    result = {'total': losses['total'].item(), 'triplet': losses['triplet'].item(), 'supcon': losses['supcon'].item(), 'optimizer_steps': int(not skipped), 'skipped_nonfinite_steps': int(skipped), 'loss_dtype': str(losses['total'].dtype), 'gradient_norm_before_clip': grad_norm.item() if finite else None, 'grad_scaler_scale': scaler.get_scale(), 'amp_enabled': amp, 'embedding_norm_min': embeddings.detach().norm(dim=1).min().item(), 'embedding_norm_max': embeddings.detach().norm(dim=1).max().item(), **{key: value if math.isfinite(value) else None for key, value in details.items()}}
    optimizer.zero_grad(set_to_none=True)
    return result

### Complete epoch, validation and history implementations

In [ ]:
def epochs_train_batch(model, batch, optimizer, scaler, device, config):
    step = steps_train_polar if config['condition'].startswith('E1-C') else steps_train_cartesian
    return step(model, batch, optimizer, scaler, device, config)

@contextmanager
def epochs_preserve_rng():
    state = state_capture_rng_state()
    try:
        yield
    finally:
        state_restore_rng_state(state)

def epochs_coverage_summary(records, seen_images, seen_irises) -> dict:
    source_counts = Counter((r['iris_id'] for r in records))
    result = {}
    for domain in config_DOMAINS:
        eligible = [r for r in records if r['dataset'] == domain and source_counts[r['iris_id']] >= 2]
        irises = {r['iris_id'] for r in eligible}
        images = {r['image_id'] for r in eligible}
        actual_images, actual_irises = (images & seen_images, irises & seen_irises)
        result[domain] = {'eligible_images': len(images), 'eligible_irises': len(irises), 'unique_images_sampled': len(actual_images), 'unique_irises_sampled': len(actual_irises), 'image_coverage': len(actual_images) / len(images), 'iris_coverage': len(actual_irises) / len(irises)}
    return result

def epochs_train_epoch(model, dataset, sampler, optimizer, scaler, device, config, epoch) -> dict:
    sampler.set_epoch(epoch - 1)
    generator = torch.Generator().manual_seed(config['seed'] + epoch)
    loader = DataLoader(dataset, batch_sampler=sampler, num_workers=0, generator=generator)
    sums = Counter(total=0.0, triplet=0.0, supcon=0.0, optimizer_steps=0, skipped_nonfinite_steps=0)
    images, irises = (set(), set())
    for batch in loader:
        metrics = epochs_train_batch(model, batch, optimizer, scaler, device, config)
        for key in sums:
            sums[key] += metrics[key]
        images.update(batch['image_id'])
        irises.update(batch['iris_id'])
    if sums['optimizer_steps'] == 0:
        raise FloatingPointError('Every optimizer step was skipped in this epoch')
    return {**{key: sums[key] / len(sampler) for key in ('total', 'triplet', 'supcon')}, 'optimizer_steps': int(sums['optimizer_steps']), 'skipped_nonfinite_steps': int(sums['skipped_nonfinite_steps']), 'batches': len(sampler), 'coverage': epochs_coverage_summary(dataset.records, images, irises)}

def epochs_infer_validation(model, dataset, pairs, device, config) -> dict:
    """No test loader exists. Evaluation disables augmentation and uses fixed pairs."""
    if not dataset.evaluation or any((row['split'] != 'validation' for row in dataset.records)):
        raise ValueError('Only deterministic frozen validation records may be evaluated')
    generator = torch.Generator().manual_seed(config['seed'])
    loader = DataLoader(dataset, batch_size=config['runtime']['validation_batch_size'], shuffle=False, num_workers=0, generator=generator)
    embeddings = {}
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            images = batch['image'].to(device, dtype=torch.float32)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=config['runtime']['amp'] and device.type == 'cuda'):
                z = model(images)
            if not torch.isfinite(z).all():
                raise FloatingPointError('Nonfinite validation embeddings')
            for image_id, embedding in zip(batch['image_id'], z.float().cpu().numpy()):
                if image_id in embeddings:
                    raise ValueError('Repeated validation image ID')
                embeddings[image_id] = embedding
    return metrics_evaluate_validation(embeddings, pairs, chunk_size=config['validation']['score_chunk_size'])

class epochs_History:
    """Atomic JSONL rewrites at epoch boundaries; rebuild curves from logged rows."""

    def __init__(self, directory: Path):
        self.directory = Path(directory)
        self.path = self.directory / 'history.jsonl'
        self.rows = [json.loads(line) for line in self.path.read_text().splitlines() if line.strip()] if self.path.exists() else []
        if [r['epoch'] for r in self.rows] != list(range(1, len(self.rows) + 1)):
            raise ValueError('Training history contains repeated/missing/out-of-order epochs')

    def reconcile(self, epoch: int, last_row: dict | None):
        self.rows = [row for row in self.rows if row['epoch'] <= epoch]
        if len(self.rows) == epoch - 1 and last_row is not None and (last_row['epoch'] == epoch):
            self.rows.append(last_row)
        if len(self.rows) != epoch:
            raise ValueError('History cannot be reconciled with last.pt')
        self._write()

    def _write(self):
        temporary = self.path.with_suffix('.jsonl.tmp')
        with temporary.open('w', encoding='utf-8', newline='\n') as stream:
            for row in self.rows:
                stream.write(json.dumps(row, sort_keys=True, allow_nan=False) + '\n')
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, self.path)

    def append(self, row):
        if row['epoch'] != len(self.rows) + 1:
            raise ValueError('History epoch is not contiguous')
        self.rows.append(row)
        self._write()

    def curves(self):
        if not self.rows:
            return
        with epochs_preserve_rng():
            matplotlib.use('Agg')
            epochs = [r['epoch'] for r in self.rows]
            fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
            for key in ('total', 'triplet', 'supcon'):
                axes[0].plot(epochs, [r['training'][key] for r in self.rows], label=key)
            axes[0].set(xlabel='Epoch', ylabel='Training loss')
            axes[0].legend()
            axes[1].plot(epochs, [r['validation']['macro_eer'] for r in self.rows], label='macro EER', linewidth=2)
            for domain in config_DOMAINS:
                axes[1].plot(epochs, [r['validation']['per_dataset'][domain]['eer'] for r in self.rows], label=domain)
            axes[1].set(xlabel='Epoch', ylabel='Validation EER')
            axes[1].legend(fontsize=7)
            temporary = self.directory / '.curves.tmp.png'
            fig.savefig(temporary, dpi=150)
            plt.close(fig)
            os.replace(temporary, self.directory / 'curves.png')

def epochs_sampler_state(sampler, epoch):
    return {'seed': sampler.seed, 'epoch': epoch - 1, 'next_epoch': epoch, 'batches_per_epoch': len(sampler), 'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24}

### Independent-seed training, best/last checkpoints and safe output handling

In [ ]:
def training_json_file(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def training_run_directory(output_root, config):
    label = 'ResNet18' if config['architecture_label'] == 'ResNet-18' else 'SwinV2_Tiny'
    return Path(output_root).expanduser().resolve() / label / config['condition'] / f"seed_{config['seed']}"

def training_membership(train_records, validation_records, pairs):
    return {'train': [r['image_id'] for r in train_records], 'validation': [r['image_id'] for r in validation_records], 'validation_pairs': {d: [(p['image_id_a'], p['image_id_b'], p['is_genuine']) for p in rows] for d, rows in pairs.items()}}

def training_save(path, model, optimizer, scaler, stopping, config, membership, row, run_id, sampler):
    payload = {'format': 'iriscuelab-notebook-v1', 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'scaler_state': scaler.state_dict(), 'epoch': row['epoch'], 'best_epoch': stopping.best_epoch, 'best_macro_validation_eer': stopping.best_score, 'early_stopping': stopping.state_dict(), 'config': config, 'membership': membership, 'rng_state': state_capture_rng_state(), 'sampler_state': epochs_sampler_state(sampler, row['epoch']), 'history_row': row, 'run_id': run_id}
    temporary = path.with_name(path.name + '.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, path)

def training_fit(config, train_records, validation_records, pairs, *, output_root, pretrained_asset=None, allow_download=False, resume=False):
    """One selected condition/seed only; no queues and no test-data access.

    Original epoch functions, RNG streams, AMP steps, validation calculations,
    selection tolerance and early stopping are reused unchanged. Only explicit
    data/output binding and the hash-free checkpoint I/O are new.
    """
    if any((r['split'] != 'train' for r in train_records)) or any((r['split'] != 'validation' for r in validation_records)):
        raise ValueError('Training accepts only frozen train and validation records')
    if {(r['dataset'], r['subject_id']) for r in train_records} & {(r['dataset'], r['subject_id']) for r in validation_records}:
        raise ValueError('Training/validation subject leakage')
    if config['sampler']['batch_size'] != 24 or config['sampler']['identities_per_domain'] != 4 or config['sampler']['captures_per_identity'] != 2:
        raise ValueError('Keep the original P=12, K=2, batch=24 protocol')
    directory = training_run_directory(output_root, config)
    if not resume and directory.exists() and any(directory.iterdir()):
        raise FileExistsError(f'New seed initialization refuses existing output: {directory}. Choose a new OUTPUT_ROOT or explicitly resume your own notebook run.')
    if resume and (not (directory / 'last.pt').is_file()):
        raise FileNotFoundError("Resume requires this notebook run's last.pt")
    device = torch.device(config['runtime']['device'])
    if device.type == 'cuda' and (not torch.cuda.is_available()):
        raise RuntimeError('CUDA was requested but is unavailable. Select the CUDA-enabled environment; CPU must be an explicit DEVICE choice.')
    directory.mkdir(parents=True, exist_ok=True)
    lock = directory / '.RUNNING.lock'
    descriptor = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY, 384)
    os.close(descriptor)
    try:
        runtime_seed_everything(config['seed'], config['runtime']['deterministic_algorithms'])
        train = data_IrisDataset(train_records, config, split='train', augment=True)
        validation = data_IrisDataset(validation_records, config, split='validation', augment=False)
        sampler = sampler_DomainBalancedPKSampler(train_records, seed=config['seed'], batches_per_epoch=config['sampler']['batches_per_epoch'], identities_per_domain=4, captures_per_identity=2)
        membership = training_membership(train_records, validation_records, pairs)
        model = models_build_model(config, pretrained=not resume, pretrained_asset=pretrained_asset if not resume else None, allow_download=allow_download).to(device)
        optimizer = runtime_build_optimizer(model, config['optimizer'])
        scaler = runtime_make_scaler(device, config['runtime']['amp'], config['runtime']['grad_scaler_init_scale'])
        schedule = config['schedule']
        stopping = state_EarlyStopping(schedule['min_epochs'], schedule['early_stopping_patience'], schedule['improvement_tolerance'])
        history, completed, run_id = (epochs_History(directory), 0, uuid.uuid4().hex)
        if resume:
            state = torch.load(directory / 'last.pt', map_location='cpu', weights_only=False)
            if state.get('format') != 'iriscuelab-notebook-v1' or state['config'] != config or state['membership'] != membership:
                raise ValueError('Resume requires the same notebook config and exact frozen image/pair membership; original archive checkpoints remain inference-only here')
            model.load_state_dict(state['model_state'], strict=True)
            optimizer.load_state_dict(state['optimizer_state'])
            scaler.load_state_dict(state['scaler_state'])
            stopping.load_state_dict(state['early_stopping'])
            completed, run_id = (state['epoch'], state['run_id'])
            history.reconcile(completed, state['history_row'])
            state_restore_rng_state(state['rng_state'])
        training_json_file(directory / 'config.json', config)
        if completed >= schedule['max_epochs'] or (completed >= schedule['min_epochs'] and stopping.bad_epochs >= stopping.patience):
            return directory
        for epoch in range(completed + 1, schedule['max_epochs'] + 1):
            begin = time.perf_counter()
            if device.type == 'cuda':
                torch.cuda.synchronize(device)
                torch.cuda.reset_peak_memory_stats(device)
            training = epochs_train_epoch(model, train, sampler, optimizer, scaler, device, config, epoch)
            metrics = epochs_infer_validation(model, validation, pairs, device, config)
            decision = stopping.update(epoch, metrics['macro_eer'])
            row = {'epoch': epoch, 'training': training, 'validation': metrics, 'learning_rate': optimizer.param_groups[0]['lr'], 'best_epoch': stopping.best_epoch, 'best_macro_eer': stopping.best_score, 'early_stopping_counter': stopping.bad_epochs, **decision, 'epoch_seconds': time.perf_counter() - begin, 'cuda_peak_allocated_bytes': torch.cuda.max_memory_allocated(device) if device.type == 'cuda' else None, 'cuda_peak_reserved_bytes': torch.cuda.max_memory_reserved(device) if device.type == 'cuda' else None}
            with epochs_preserve_rng():
                training_save(directory / 'last.pt', model, optimizer, scaler, stopping, config, membership, row, run_id, sampler)
                if decision['improved']:
                    training_save(directory / 'best.pt', model, optimizer, scaler, stopping, config, membership, row, run_id, sampler)
                history.append(row)
                if config['logging']['write_curves_every_epoch']:
                    history.curves()
                training_json_file(directory / 'run.json', {'architecture': config['architecture_label'], 'condition': config['condition'], 'seed': config['seed'], 'run_id': run_id, 'best_epoch': stopping.best_epoch, 'best_macro_validation_eer': stopping.best_score, 'completed_epoch': epoch, 'status': 'early_stopped' if decision['should_stop'] else 'max_epochs_reached' if epoch == schedule['max_epochs'] else 'training'})
            print(f"{config['architecture_label']} {config['condition']} seed={config['seed']} epoch={epoch} loss={training['total']:.6f} val_EER={100 * metrics['macro_eer']:.4f}% best={stopping.best_epoch} patience={stopping.bad_epochs}", flush=True)
            if decision['should_stop']:
                break
        del model, optimizer, scaler
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        return directory
    finally:
        lock.unlink(missing_ok=True)

## 3. Frozen configuration and software checks

In [ ]:
configurations = [config_frozen_config(ARCHITECTURE, CONDITION, seed, DEVICE) for seed in SEEDS]
config = configurations[0]
display(pd.Series(config_environment_info(), name="Environment"))
display(pd.Series({key:config[key] for key in ["architecture","input_shape","normalization","augmentation","sampler","objective","optimizer","schedule"]}))
assert SEEDS == [42,123,2025]
assert config["sampler"]["batch_size"] == 24
assert config["schedule"]["selection"] == "macro_validation_eer"

## 4. Load and validate train/validation assets

In [ ]:
if CHECK_REAL_ASSETS or RUN_TRAINING:
    records, inventory_audit = data_load_inventory(IMAGE_MANIFEST, data_root=DATA_ROOT,
        subject_splits=SUBJECT_SPLITS, image_exclusions=IMAGE_EXCLUSIONS, path_remap=PATH_REMAP)
    train_records, train_unavailable = data_condition_view(records, CONDITION, "train")
    validation_records, validation_unavailable = data_condition_view(records, CONDITION, "validation")
    validation_pairs = pairs_load_validation_pairs(VALIDATION_PAIR_FILES, validation_records, CONDITION,
        original_validation_records=[r for r in records if r["split"]=="validation"])
    print("Train images:",len(train_records),"PK-eligible:",len(data_pk_eligible(train_records)))
    print("Unavailable train/validation:",len(train_unavailable),len(validation_unavailable))
    sampler = sampler_DomainBalancedPKSampler(train_records, seed=42,
        batches_per_epoch=config["sampler"]["batches_per_epoch"],identities_per_domain=4,captures_per_identity=2)
    display(pd.DataFrame(sampler.eligibility_summary).T)
    print("Validation genuine/impostor:",{d:(sum(p["is_genuine"] for p in rows),sum(p["pair_type"]=="impostor" for p in rows)) for d,rows in validation_pairs.items()})
else:
    print("Real asset access disabled. Configure paths and enable CHECK_REAL_ASSETS or RUN_TRAINING.")

## 5. Optional pretrained model smoke check

In [ ]:
if RUN_MODEL_SMOKE:
    runtime_seed_everything(42,config["runtime"]["deterministic_algorithms"])
    model = models_build_model(config,pretrained=True,pretrained_asset=PRETRAINED_ASSET,allow_download=ALLOW_PRETRAINED_DOWNLOAD).to(DEVICE).eval()
    with torch.inference_mode():
        z=model(torch.zeros(1,*config["input_shape"],device=DEVICE))
    assert z.shape==(1,256) and torch.isfinite(z).all()
    assert torch.allclose(z.norm(dim=1),torch.ones(1,device=DEVICE),atol=1e-5)
    display(pd.Series(models_model_summary(model,config)))
    del model,z
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

## 6. Train this condition: seeds 42, 123 and 2025

P=12, K=2, four distinct subjects per domain, two distinct captures per physical eye. Cross-domain and opposite-eye pairs are absent from both loss denominators and hard-negative mining.
Triplet margin 0.3, SupCon temperature 0.07 and weight 0.5; AdamW 1e−4/1e−4, clipping 5.0.
The learning rate is constant. Max/min epochs 150/30, patience 25; strict validation macro-EER improvement greater than 0.0001.
CUDA OOM propagates without smaller batches or gradient accumulation. Best and last checkpoints are independent for each seed.
New runs refuse nonempty output. Resume is supported only for checkpoints produced by this notebook workflow.

In [ ]:
training_directories=[]
if RUN_TRAINING:
    for seed_config in configurations:
        training_directories.append(training_fit(seed_config,train_records,validation_records,validation_pairs,
            output_root=OUTPUT_ROOT,pretrained_asset=PRETRAINED_ASSET,allow_download=ALLOW_PRETRAINED_DOWNLOAD,
            resume=RESUME_NOTEBOOK_RUNS))
else:
    print("Training disabled; no optimizer steps or recognition evaluation run by default.")

## 7. Inspect actual completed histories and matched paper results

In [ ]:
if SHOW_RECORDED_RESULTS:
    history_root=REPOSITORY_ROOT / 'SwinV2-Tiny' / "Training Histories" / CONDITION
    results_root=REPOSITORY_ROOT / 'SwinV2-Tiny' / "Results" / CONDITION
    if not history_root.is_dir():
        raise FileNotFoundError("Set REPOSITORY_ROOT to view saved records, or disable SHOW_RECORDED_RESULTS for a copied standalone notebook.")
    print("RECORDED training/validation, not reproduced in this notebook session")
    display(pd.read_csv(history_root / "runs.csv"))
    display(pd.read_csv(history_root / "individual_metrics.csv"))
    for seed in SEEDS:
        display(Image(filename=str(history_root / f"seed_{seed}/curves.png")))
    print("RECORDED matched-population paper evaluation; all five representations share each dataset's exact members/pairs")
    display(pd.read_csv(results_root / "individual_metrics.csv"))
    display(pd.read_csv(results_root / "three_seed_summary.csv"))

## 8. New outputs and independent evaluation

New histories, curves, configuration and best/last checkpoints are written under `OUTPUT_ROOT`, never to the recorded history folders.
To plot a new run: `epochs_History(training_directories[0]).curves()`.
The matched frozen-checkpoint recognition implementation is fully contained in [E2-A](../../E2%20Experiments/E2-A/E2-A.ipynb).
It accepts this notebook's original-compatible checkpoints directly; it does not depend on executing this notebook.
No model selection, threshold fitting or subject reassignment uses test or external datasets.